In [1]:
# Make data for learning analysis

import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
from matplotlib import colors as mcolors
from matplotlib import patches as mpatches
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

np.seterr(divide='ignore', invalid='ignore') # ignore divide by zero warnings

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11
#plt.rcParams['text.usetex'] = True

from sklearn.decomposition import PCA

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
with open("../../config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import writing_tools as wt
import utils
#import emb

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False
REPLACE_ANALYSIS_DATA = False

ANALYSIS_DATA_FILEPATH = os.path.join(DATA_PATH, "learning_analysis_data.parquet")


In [2]:
df = pd.read_parquet(ANALYSIS_DATA_FILEPATH)

In [3]:
#res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/04_learning.R"], capture_output=True, text=True, check=True)
#print(res.stdout)

In [4]:
coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "learning_analysis_coefs.parquet"))

In [6]:
# output table

header = r"""\begin{table}[H]
\centering
\caption{Effect of Observed Tip Feedback on Post Type Choice}
\vspace{0.2cm}
\label{tbl_learning}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lcccccc}
\toprule
 & \multicolumn{6}{c}{Observed feedback weighted by:} \\
 & \multicolumn{2}{c}{Uniform} & \multicolumn{2}{c}{Recency} & \multicolumn{2}{c}{\makecell{Distance to user's\\peak activity}} \\ [1ex]
 & (1) & (2) & (3) & (4) & (5) & (6) \\
\midrule
 &  &  &  &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:} This table reports estimated coefficients for the conditional logit model of post type choice described in Section \ref{sec_v4v} equation \eqref{eq_learning}. Each unit of observation is a user $i$'s choice to make a post of type $c$ at time $t$. The user chooses the post type based on their prior observation of tips earned by posts of each type. ``Observed Tips'' is a weighted average of ln(sats) earned on posts of each type $c$ prior to time $t$. See Section \ref{sec_v4v} for a description of the model and weighting functions. ``User's Experience'' is the $\ln$ of the number of posts the user $i$ has made up to time $t$. ``User Excess Tips'' is the weighted average of the user's own ln(sats) for each post type $c$ earned in excess of the overall weighted average for that post type.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""

model_order = ['r0', 'r3', 'r1', 'r4', 'r2', 'r5']

vars = [
    ("signal", r"Observed Tips"),
    ("signal_x_exp", r"Observed Tips $\times$ User's Experience"),
    ("surpr", r"User Excess Tips")
]


tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in model_order:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in model_order:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += " & "*len(model_order) + r" \\" + "\n"
tbl += r"Post Type FE" + " & Y "*len(model_order) + r" \\" + "\n"
tbl += r"Post Type $\times$ Linear Time Trend" + " & Y "*len(model_order) + r" \\" + "\n"
tbl += " & "*len(model_order) + r" \\" + "\n"

tbl += "Observations "
for rn in model_order:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ " + "\n"

tbl += "McFadden's Pseudo-R$^2$"
for rn in model_order:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="pseudo_r2")
    if idx.sum()==0:
        tbl += " & "
    else:
        tbl += f" & {coefs_df.loc[idx, "estimate"].values[0]:.3f}"
tbl += r" \\ [1.8ex]" + "\n"

print(header + tbl + footer)

with open(os.path.join(LOCAL_PATH, "results", "tbl_learning.tex"), "w") as f:
    f.write(header + tbl + footer)


\begin{table}[H]
\centering
\caption{Effect of Observed Tip Feedback on Post Type Choice}
\vspace{0.2cm}
\label{tbl_learning}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lcccccc}
\toprule
 & \multicolumn{6}{c}{Observed feedback weighted by:} \\
 & \multicolumn{2}{c}{Uniform} & \multicolumn{2}{c}{Recency} & \multicolumn{2}{c}{\makecell{Distance to user's\\peak activity}} \\ [1ex]
 & (1) & (2) & (3) & (4) & (5) & (6) \\
\midrule
 &  &  &  &  &  &  \\
Observed Tips  & 0.844$^{***}$ & 0.905$^{***}$ & 0.980$^{***}$ & 0.991$^{***}$ & 0.819$^{***}$ & 0.881$^{***}$ \\
 & (0.018) & (0.018) & (0.017) & (0.017) & (0.016) & (0.017) \\ [1.8ex]
Observed Tips $\times$ User's Experience  & -0.187$^{***}$ & -0.181$^{***}$ & -0.207$^{***}$ & -0.206$^{***}$ & -0.177$^{***}$ & -0.173$^{***}$ \\
 & (0.002) & (0.002) & (0.002) & (0.002) & (0.002) & (0.002) \\ [1.8ex]
User Excess Tips  &  & 0.123$^{***}$ &  & 0.019$^{***}$ &  & 0.117$^{***}$ \\
 &  & (0.003) &  & (0.003) & 